In [1]:
import pandas as pd 
import numpy as np 

In [2]:
df = pd.read_csv('../../../data/interim/haryana_cleaned_groundwater.csv')

In [3]:
df.shape

(745175, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 745175 entries, 0 to 745174
Data columns (total 8 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   Station                                       745175 non-null  object 
 1   State                                         745175 non-null  object 
 2   District LGD Code                             745175 non-null  int64  
 3   District                                      745175 non-null  object 
 4   Latitude                                      745175 non-null  float64
 5   Longitude                                     745175 non-null  float64
 6   Data Acquisition Time                         745175 non-null  object 
 7   Groundwater Level Telemetry 6 Hourly (meter)  745175 non-null  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 45.5+ MB


In [5]:
# Date-Time Conversion:

df["Data Acquisition Time"] = pd.to_datetime(df["Data Acquisition Time"]) 
df.dtypes

Station                                                 object
State                                                   object
District LGD Code                                        int64
District                                                object
Latitude                                               float64
Longitude                                              float64
Data Acquisition Time                           datetime64[ns]
Groundwater Level Telemetry 6 Hourly (meter)           float64
dtype: object

In [6]:
#F-1: Calendar-Based Features: 

df["Year"] = df["Data Acquisition Time"].dt.year
df["Month"] = df["Data Acquisition Time"].dt.month
df["Day"] = df["Data Acquisition Time"].dt.day
df["Day_of_Week"] = df["Data Acquisition Time"].dt.dayofweek
df["Week_of_Year"] = df["Data Acquisition Time"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Data Acquisition Time"].dt.quarter
df['Is_Weekend'] = (df['Day_of_Week'] >= 5).astype(int) 

In [7]:
df[
    [
        "Data Acquisition Time",
        "Year",
        "Month",
        "Day",
        "Day_of_Week",
        "Week_of_Year",
        "Quarter",
        "Is_Weekend",
    ]
].head()

,Data Acquisition Time,Year,Month,Day,Day_of_Week,Week_of_Year,Quarter,Is_Weekend
0,2022-07-13 18:00:00,2022,7,13,2,28,3,0
1,2022-07-14 00:00:00,2022,7,14,3,28,3,0
2,2022-07-14 06:00:00,2022,7,14,3,28,3,0
3,2022-07-14 12:00:00,2022,7,14,3,28,3,0
4,2022-07-14 18:00:00,2022,7,14,3,28,3,0


In [8]:
#F-2: Season Feature: 

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

df["Season"] = df["Month"].apply(get_season)
df['Season']

0         Monsoon
1         Monsoon
2         Monsoon
3         Monsoon
4         Monsoon
           ...   
745170    Monsoon
745171    Monsoon
745172    Monsoon
745173    Monsoon
745174    Monsoon
Name: Season, Length: 745175, dtype: object

In [9]:
# now since the data contains measurements from multiple stations. If we create lag features directly, the previous row for one station could come from a completely different station, which would produce incorrect values 
#so we will sort the data on the basis of 'Station' and 'Data Acquisition Time' 

df = df.sort_values(
    by=['Station', 'Data Acquisition Time']
).reset_index(drop=True)

In [10]:
df[['Station', 'Data Acquisition Time']].head(20)

,Station,Data Acquisition Time
0,AHMADPUR MAJRA_1,2022-11-10 18:00:00
1,AHMADPUR MAJRA_1,2022-11-11 00:00:00
2,AHMADPUR MAJRA_1,2022-11-11 06:00:00
3,AHMADPUR MAJRA_1,2022-11-11 12:00:00
4,AHMADPUR MAJRA_1,2022-11-11 18:00:00
5,AHMADPUR MAJRA_1,2022-11-12 00:00:00
6,AHMADPUR MAJRA_1,2022-11-12 06:00:00
7,AHMADPUR MAJRA_1,2022-11-12 12:00:00
8,AHMADPUR MAJRA_1,2022-11-12 18:00:00
9,AHMADPUR MAJRA_1,2022-11-13 00:00:00


In [11]:
#F-3: Lag Features: 
# Groundwater levels are highly autocorrelated, so previous values are often the strongest predictors of future values 

In [12]:
#Lag-1: 

df['GW_Lag_1'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(1)
)
df['GW_Lag_1']

0            NaN
1         -6.162
2         -6.131
3         -6.106
4         -6.081
           ...  
745170   -23.126
745171   -23.149
745172   -23.293
745173   -23.223
745174   -23.086
Name: GW_Lag_1, Length: 745175, dtype: float64

In [14]:
#Lag-2: 

df['GW_Lag_2'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(2)
) 
df['GW_Lag_2']

0            NaN
1            NaN
2         -6.162
3         -6.131
4         -6.106
           ...  
745170   -23.227
745171   -23.126
745172   -23.149
745173   -23.293
745174   -23.223
Name: GW_Lag_2, Length: 745175, dtype: float64

In [16]:
#Lag-3: 

df['GW_Lag_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .shift(3)
) 
df['GW_Lag_3']

0            NaN
1            NaN
2            NaN
3         -6.162
4         -6.131
           ...  
745170   -23.292
745171   -23.227
745172   -23.126
745173   -23.149
745174   -23.293
Name: GW_Lag_3, Length: 745175, dtype: float64

In [17]:
#F-4: Rolling Statistics Features: 

In [19]:
# Rolling Mean (Window = 3)

df['GW_Rolling_Mean_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.rolling(window=3).mean())
)
df['GW_Rolling_Mean_3']

0               NaN
1               NaN
2         -6.133000
3         -6.106000
4         -6.059333
            ...    
745170   -23.167333
745171   -23.189333
745172   -23.221667
745173   -23.200667
745174   -23.144000
Name: GW_Rolling_Mean_3, Length: 745175, dtype: float64

In [21]:
# Rolling Standard Deviation (Window = 3) 

df['GW_Rolling_STD_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.rolling(window=3).std())
)
df['GW_Rolling_STD_3']

0              NaN
1              NaN
2         0.028054
3         0.025000
4         0.060484
            ...   
745170    0.052937
745171    0.090512
745172    0.072009
745173    0.105292
745174    0.070873
Name: GW_Rolling_STD_3, Length: 745175, dtype: float64

In [23]:
df['GW_Expanding_Mean'] = (
    df.groupby('Station')['Groundwater Level Telemetry 6 Hourly (meter)']
      .transform(lambda x: x.expanding().mean())
)
df['GW_Expanding_Mean']

0         -6.162000
1         -6.146500
2         -6.133000
3         -6.120000
4         -6.094200
            ...    
745170   -20.771025
745171   -20.771611
745172   -20.772181
745173   -20.772719
745174   -20.773265
Name: GW_Expanding_Mean, Length: 745175, dtype: float64

This feature answers:

"What has been the average groundwater level at this station up to the current observation?"

Unlike a rolling mean, which only looks at the last few observations, the expanding mean summarizes the station's historical behavior.

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 745175 entries, 0 to 745174
Data columns (total 22 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   Station                                       745175 non-null  object        
 1   State                                         745175 non-null  object        
 2   District LGD Code                             745175 non-null  int64         
 3   District                                      745175 non-null  object        
 4   Latitude                                      745175 non-null  float64       
 5   Longitude                                     745175 non-null  float64       
 6   Data Acquisition Time                         745175 non-null  datetime64[ns]
 7   Groundwater Level Telemetry 6 Hourly (meter)  745175 non-null  float64       
 8   Year                                          745175 n

In [25]:
df.to_csv('../../../data/processed/haryana_featured_groundwater.csv')